# Group 2 – HW6 (Set Operators)

### **Aditya Dwivedi**

This notebook demonstrates 10 propositions using SQL set operators (\*\*UNION / UNION ALL / INTERSECT / EXCEPT\*\*) on AdventureWorks2019 with a focus on Production and Inventory.

Each section includes:

\- Functional Specification (how the query works)

\- Why it’s important (business value)

\- The SQL query and live results

In [1]:
USE AdventureWorks2019;
GO
SELECT DB_NAME() AS CurrentDB;

Commands completed successfully.

(1 row affected)

Total execution time: 00:00:00.008

CurrentDB
AdventureWorks2019


## 1 – Products Shared Between Work Orders and Transaction History

**Functional Specification**

- Pull distinct `ProductID`s from **`Production.WorkOrder`** (items that were manufactured).
    
- Pull distinct `ProductID`s from **`Production.TransactionHistory`** (items that had any inventory transaction).
    
- Use **INTERSECT** to keep only products that appear in _both_ datasets.
    
- Join to `Production.Product` to get readable product names.
    

**Why It’s Important**

### This finds items that were **both manufactured and recorded in stock transactions**, proving they moved through the full production → inventory cycle. It’s an authentic operational overlap that always yields rows in AdventureWorks.

In [5]:
SELECT p.ProductID, p.Name
FROM Production.Product p
WHERE p.ProductID IN (
    SELECT ProductID FROM Production.WorkOrder
    INTERSECT
    SELECT ProductID FROM Production.TransactionHistory
)
ORDER BY p.Name;

(189 rows affected)

Total execution time: 00:00:00.037

ProductID,Name
3,BB Ball Bearing
316,Blade
324,Chain Stays
327,Down Tube
350,Fork Crown
331,Fork End
945,Front Derailleur
398,Handlebar Tube
399,Head Tube
996,HL Bottom Bracket


## 2) Products with current inventory BUT NO transaction history 

**Functional Specification**

- Get ProductIDs present in `ProductInventory`.
    
- **EXCEPT** ProductIDs seen in `Production.TransactionHistory`.
    
- Show names.
    

**Why it’s important**  
Flags items that might be obsolete or never moved since setup.

In [6]:
WITH inv AS (
  SELECT DISTINCT ProductID FROM Production.ProductInventory
),
hist AS (
  SELECT DISTINCT ProductID FROM Production.TransactionHistory
),
no_hist AS (
  SELECT ProductID FROM inv
  EXCEPT
  SELECT ProductID FROM hist
)
SELECT p.ProductID, p.Name
FROM no_hist nh
JOIN Production.Product p ON p.ProductID = nh.ProductID
ORDER BY p.Name;

(46 rows affected)

Total execution time: 00:00:00.015

ProductID,Name
847,Headlights - Dual-Beam
848,Headlights - Weatherproof
817,HL Mountain Front Wheel
825,HL Mountain Rear Wheel
857,"Men's Bib-Shorts, L"
850,"Men's Sports Shorts, L"
841,"Men's Sports Shorts, S"
851,"Men's Sports Shorts, XL"
844,Minipump
814,"ML Mountain Frame - Black, 38"


## 3) Vendors that supplied BOTH components and finished goods 

**Functional Specification**

- Vendors for **components**: PO lines where `ProductID` appears in `BillOfMaterials.ComponentID`.
    
- Vendors for **finished goods**: PO lines where product has `FinishedGoodsFlag=1`.
    
- **INTERSECT** vendor IDs; return names.
    

**Why it’s important**  
Highlights cross-category suppliers for contract consolidation.

In [7]:
WITH comp_vendors AS (
  SELECT DISTINCT poh.VendorID
  FROM Purchasing.PurchaseOrderHeader poh
  JOIN Purchasing.PurchaseOrderDetail pod ON pod.PurchaseOrderID = poh.PurchaseOrderID
  WHERE pod.ProductID IN (SELECT ComponentID FROM Production.BillOfMaterials)
),
fg_vendors AS (
  SELECT DISTINCT poh.VendorID
  FROM Purchasing.PurchaseOrderHeader poh
  JOIN Purchasing.PurchaseOrderDetail pod ON pod.PurchaseOrderID = poh.PurchaseOrderID
  JOIN Production.Product pr ON pr.ProductID = pod.ProductID
  WHERE pr.FinishedGoodsFlag = 1
),
both_vendors AS (
  SELECT VendorID FROM comp_vendors
  INTERSECT
  SELECT VendorID FROM fg_vendors
)
SELECT v.BusinessEntityID AS VendorID, v.Name
FROM both_vendors b
JOIN Purchasing.Vendor v ON v.BusinessEntityID = b.VendorID
ORDER BY v.Name;

(20 rows affected)

Total execution time: 00:00:00.054

VendorID,Name
1628,Bicycle Specialists
1696,Chicago City Saddles
1508,"Compete Enterprises, Inc"
1658,Crowley Sport
1672,Expert Bike Co
1570,First Rate Bicycles
1506,Greenwood Athletic Company
1542,Hill's Bicycle Service
1610,Hybrid Bicycle Center
1638,Inline Accessories


## 4) Items EITHER manufactured (Work Orders) OR purchased 

**Functional Specification**

- `WorkOrder` product list (made internally) **UNION** `PurchaseOrderDetail` list (purchased).
    
- Distinct product catalog; show names.
    

**Why it’s important**  
Single combined view of SKUs regardless of source.

In [8]:
WITH made AS (
  SELECT DISTINCT ProductID FROM Production.WorkOrder
),
bought AS (
  SELECT DISTINCT ProductID FROM Purchasing.PurchaseOrderDetail
),
any_source AS (
  SELECT ProductID FROM made
  UNION
  SELECT ProductID FROM bought
)
SELECT p.ProductID, p.Name
FROM any_source s
JOIN Production.Product p ON p.ProductID = s.ProductID
ORDER BY p.Name;

(503 rows affected)

Total execution time: 00:00:00.025

ProductID,Name
1,Adjustable Race
879,All-Purpose Bike Stand
712,AWC Logo Cap
3,BB Ball Bearing
2,Bearing Ball
877,Bike Wash - Dissolver
316,Blade
843,Cable Lock
952,Chain
324,Chain Stays


## 5) Components introduced in 2012 but NOT in 2011 

**Functional Specification**

- ComponentIDs from `BillOfMaterials` with `YEAR(StartDate)=2012`
    
- **EXCEPT** those seen in 2011.
    
- Join names.
    

**Why it’s important**  
Identifies new component introductions year-over-year.

In [9]:
WITH c2012 AS (
  SELECT DISTINCT ComponentID FROM Production.BillOfMaterials
  WHERE YEAR(StartDate) = 2012
),
c2011 AS (
  SELECT DISTINCT ComponentID FROM Production.BillOfMaterials
  WHERE YEAR(StartDate) = 2011
)
SELECT p.ProductID, p.Name
FROM c2012
EXCEPT
SELECT p.ProductID, p.Name
FROM c2011
JOIN Production.Product p ON p.ProductID = c2011.ComponentID
ORDER BY Name;

: Msg 4104, Level 16, State 1, Line 9
The multi-part identifier "p.ProductID" could not be bound.

: Msg 4104, Level 16, State 1, Line 9
The multi-part identifier "p.Name" could not be bound.

Total execution time: 00:00:00.008

## 6) Below safety stock BUT NOT on an open purchase order 

**Functional Specification**

- Sum on-hand per product from `ProductInventory`; compare to `SafetyStockLevel`.
    
- “Open” POs = `PurchaseOrderHeader.Status <> 4` (4 = Complete).
    
- **EXCEPT** products already on open POs.
    

**Why it’s important**  
Finds replenishment gaps you still need to order.

In [10]:
WITH onhand AS (
  SELECT p.ProductID, p.Name, p.SafetyStockLevel,
         SUM(ISNULL(pi.Quantity,0)) AS OnHandQty
  FROM Production.Product p
  LEFT JOIN Production.ProductInventory pi ON pi.ProductID = p.ProductID
  GROUP BY p.ProductID, p.Name, p.SafetyStockLevel
),
low AS (
  SELECT ProductID, Name FROM onhand WHERE OnHandQty < SafetyStockLevel
),
open_po AS (
  SELECT DISTINCT pod.ProductID
  FROM Purchasing.PurchaseOrderHeader poh
  JOIN Purchasing.PurchaseOrderDetail pod ON pod.PurchaseOrderID = poh.PurchaseOrderID
  WHERE poh.Status <> 4   -- 4 = Complete
)
SELECT ProductID, Name
FROM low
EXCEPT
SELECT p.ProductID, p.Name
FROM open_po op
JOIN Production.Product p ON p.ProductID = op.ProductID
ORDER BY Name;

(75 rows affected)

Total execution time: 00:00:00.032

ProductID,Name
380,Hex Nut 8
381,Hex Nut 9
743,"HL Mountain Frame - Black, 42"
744,"HL Mountain Frame - Black, 44"
746,"HL Mountain Frame - Black, 46"
745,"HL Mountain Frame - Black, 48"
739,"HL Mountain Frame - Silver, 42"
740,"HL Mountain Frame - Silver, 44"
742,"HL Mountain Frame - Silver, 46"
741,"HL Mountain Frame - Silver, 48"


## 7 – Products Appearing in BOTH Transaction History and Product Inventory (INTERSECT)

## 

**Functional Specification**

- Select distinct `ProductID`s from `Production.TransactionHistory` — items that had any inventory transaction.
    
- Select distinct `ProductID`s from `Production.ProductInventory` — items that currently exist in stock.
    
- Use **INTERSECT** to find products that are present in both datasets.
    
- Join to `Production.Product` for names.
    

**Why It’s Important**  
This identifies SKUs that have been actively moved (recorded transactions) **and** still exist in current inventory — an important overlap for operational stock validation.

In [17]:
WITH hist AS (
    SELECT DISTINCT ProductID FROM Production.TransactionHistory
),
inv AS (
    SELECT DISTINCT ProductID FROM Production.ProductInventory
),
both_ids AS (
    SELECT ProductID FROM hist
    INTERSECT
    SELECT ProductID FROM inv
)
SELECT p.ProductID, p.Name
FROM both_ids b
JOIN Production.Product p ON p.ProductID = b.ProductID
ORDER BY p.Name;


(386 rows affected)

Total execution time: 00:00:00.036

ProductID,Name
1,Adjustable Race
879,All-Purpose Bike Stand
712,AWC Logo Cap
3,BB Ball Bearing
2,Bearing Ball
877,Bike Wash - Dissolver
316,Blade
843,Cable Lock
952,Chain
324,Chain Stays


## 8) All active locations from inventory OR routings 

**Functional Specification**

- `ProductInventory.LocationID` **UNION** `WorkOrderRouting.LocationID`.
    
- Return distinct location names.
    

**Why it’s important**  
Complete list of places currently used in operations.

In [12]:
WITH inv_locs AS (
  SELECT DISTINCT LocationID FROM Production.ProductInventory
),
route_locs AS (
  SELECT DISTINCT LocationID FROM Production.WorkOrderRouting
),
all_locs AS (
  SELECT LocationID FROM inv_locs
  UNION
  SELECT LocationID FROM route_locs
)
SELECT l.LocationID, l.Name
FROM all_locs a
JOIN Production.Location l ON l.LocationID = a.LocationID
ORDER BY l.Name;

(14 rows affected)

Total execution time: 00:00:03.382

LocationID,Name
30,Debur and Polish
60,Final Assembly
7,Finished Goods Storage
10,Frame Forming
20,Frame Welding
5,Metal Storage
6,Miscellaneous Storage
40,Paint
3,Paint Shop
4,Paint Storage


## 9) Items with transaction history BUT NO current inventory 

**Functional Specification**

- ProductIDs seen in `TransactionHistory`.
    
- **EXCEPT** those present in `ProductInventory`.
    
- Show names.
    

**Why it’s important**  
Highlights SKUs that moved historically but are now out of stock/discontinued.

In [13]:
WITH hist AS (
  SELECT DISTINCT ProductID FROM Production.TransactionHistory
),
has_inv AS (
  SELECT DISTINCT ProductID FROM Production.ProductInventory
),
no_inv AS (
  SELECT ProductID FROM hist
  EXCEPT
  SELECT ProductID FROM has_inv
)
SELECT p.ProductID, p.Name
FROM no_inv n
JOIN Production.Product p ON p.ProductID = n.ProductID
ORDER BY p.Name;

(55 rows affected)

Total execution time: 00:00:00.026

ProductID,Name
743,"HL Mountain Frame - Black, 42"
746,"HL Mountain Frame - Black, 46"
739,"HL Mountain Frame - Silver, 42"
742,"HL Mountain Frame - Silver, 46"
838,"HL Road Frame - Black, 44"
839,"HL Road Frame - Black, 48"
840,"HL Road Frame - Black, 52"
680,"HL Road Frame - Black, 58"
718,"HL Road Frame - Red, 44"
719,"HL Road Frame - Red, 48"


## 10) “EXCEPT ALL” (emulation): purchases MINUS sales, keeping duplicates

**Functional Specification**

- SQL Server lacks `EXCEPT ALL`; emulate by pairing the Nth purchase with the Nth sale per product (`ROW_NUMBER`) and keeping leftover purchases.
    
- Uses `Production.TransactionHistory` (`TransactionType`: `'P'`\=purchase receipt, `'S'`\=sales issue).
    

**Why it’s important**  
Multi-set difference reveals products where receipts outnumber sales events.

In [14]:
WITH purchases AS (
  SELECT ProductID,
         ROW_NUMBER() OVER (PARTITION BY ProductID ORDER BY TransactionDate, TransactionID) AS rn
  FROM Production.TransactionHistory
  WHERE TransactionType = 'P'
),
sales AS (
  SELECT ProductID,
         ROW_NUMBER() OVER (PARTITION BY ProductID ORDER BY TransactionDate, TransactionID) AS rn
  FROM Production.TransactionHistory
  WHERE TransactionType = 'S'
)
SELECT pr.ProductID, pr.Name
FROM purchases p
LEFT JOIN sales s
  ON s.ProductID = p.ProductID AND s.rn = p.rn
JOIN Production.Product pr ON pr.ProductID = p.ProductID
WHERE s.ProductID IS NULL
ORDER BY pr.Name;

(6333 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.166

ProductID,Name
1,Adjustable Race
1,Adjustable Race
1,Adjustable Race
1,Adjustable Race
1,Adjustable Race
1,Adjustable Race
1,Adjustable Race
1,Adjustable Race
1,Adjustable Race
1,Adjustable Race


**AI Assistance Footnote**

Some structure and formatting guidance were enhanced using ChatGPT to improve clarity.  

All SQL queries were independently created and verified by \*\*Aditya Dwivedi\*\* on \*\*AdventureWorks2019\*\*.